# Gene Centric indexer
First attempt at gene indexing.
Works, although very inefficiently
```
gene{}
     |___ case[]
             |___ ssm[]
                   |___ consequence[]
                   |             |_____ transcript{}
                   |                          |_____ annotation{}
                   |___ observation[]
```

In [1]:
import os
import requests
import uuid
%load_ext autoreload
from exports.mappings import GeneMapper, SSMMapper, Mapper
from exports.utils import get_array_paths

from pyspark.sql.functions import col, max, collect_list, size, sum, first, struct, udf, regexp_extract, lit, count, broadcast
from pyspark.sql.types import StringType

## Load combined maf into spark

In [2]:
url = 's3a://test/combined_mafs.csv'
    
df = sqlContext.read.format('com.databricks.spark.csv')\
                .options(header='true', inferschema='true')\
                .load(url)\
                .drop_duplicates()
        

In [3]:
#df = df.limit(500000)

In [4]:
'''
df = sqlContext.read.format('com.databricks.spark.csv')\
                .options(header='true', inferschema='true', comment='#', delimiter='\t')\
                .load('/home/ubuntu/tests/data/test.maf')
'''        

"\ndf = sqlContext.read.format('com.databricks.spark.csv')                .options(header='true', inferschema='true', comment='#', delimiter='\t')                .load('/home/ubuntu/tests/data/test.maf')\n"

## Rename and select desired columns in the mafs

In [5]:
%autoreload
from exports.utils import (
    maf_annotation_map,
    maf_gene_map,
    maf_observation_map,
    maf_ssm_map,
    maf_transcript_map,
    tumor_genotype_map,
    tumor_validation_map,
    normal_genotype_map,
    sample_map,
    input_bam_map,
    read_depth_map,
    maf_cols
)

maf_df = df.select(*( col(v).alias(k) for k,v in maf_cols.items() ))

## Augment maf df by extracting submitter_id and creating ssm_uuids

In [6]:
maf_df = maf_df.withColumn('_case_submitter_id',
                           regexp_extract(col('tumor_sample_barcode'),
                                          '([A-Z]{4}-[A-Z0-9]{2}-[A-Z0-9]{4})',1))
maf_ssm_map.update({'_case_submitter_id':'_case_submitter_id'})

In [7]:
def ssm_uuid(chromosome, start_position, ref_allele, tumor_allele):
    '''
    SNP: "{chromosome}:g.{start_position}{reference_allele}>{tumor_allele}"
    DEL: "{chromosome}:g.{start_position}del{reference_allele}"
    INS: "{chromosome}:g.{start_position}_{end_position}ins{tumor_allele}"
    '''
    chromosome = chromosome.replace('chr','')
    label = '{}:g.{}:{}>{}'.format(chromosome, start_position, ref_allele, tumor_allele)
    return str(uuid.uuid5(uuid.UUID('d15296a3-38ed-412e-8ace-75e235f82f55'), label))

ssm_uuid_udf =udf(ssm_uuid, StringType())
maf_df = maf_df.withColumn('ssm_id', ssm_uuid_udf(col('chromosome'), col('start_position'), col('reference_allele'), col('tumor_allele')))
maf_observation_map.update({'ssm_id':'ssm_id'})
maf_ssm_map.update({'ssm_id':'ssm_id'})

## Slice and dice until we get to the format we want

### Gene df

In [8]:
gene_df = maf_df.select(*( col(k) for k in maf_gene_map.keys() + ['_case_submitter_id'] ))
# Fill in empty data we don't know about
gene_df = gene_df.withColumn('description', lit(None).cast(StringType()))\
                 .drop_duplicates()

In [9]:
gene_df.columns

['gene_chromosome',
 'biotype',
 'gene_start',
 'gene_end',
 'canonical_transcript_id',
 'symbol',
 'gene_id',
 'name',
 '_case_submitter_id',
 'description']

### SSM df

In [10]:
ssm_df = maf_df.select(*( col(k) for k in maf_ssm_map.keys()))\
               .drop_duplicates()
#ssm_df.printSchema()

### Transcript-annotation df

```
transcript{}
     |_____ annotation{}
```

In [11]:
# Rename columns
tran_anno_df = maf_df.select('ssm_id',*( maf_transcript_map.keys() + maf_annotation_map.keys() ))
# Select annotation into nested format
tran_anno_df = tran_anno_df.select(struct(*maf_annotation_map.keys()).alias('annotation'), 'ssm_id', *maf_transcript_map.keys())\
                           .drop_duplicates()
#tran_anno_df.printSchema()

In [12]:
#ltran_df = maf_df.select('gene_id', *maf_transcript_map.keys())
#gene_df = gene_df.join(tran_df, gene_df.gene_id == tran_df.gene_id, 'left')

In [13]:
#gene_df.printSchema()

### Observation df

In [14]:
observation_df = maf_df.select(*(maf_observation_map.keys()
                                 +normal_genotype_map.keys()
                                 +tumor_genotype_map.keys()
                                 +tumor_validation_map.keys()
                                 +read_depth_map.keys()
                                 +input_bam_map.keys()
                                 +sample_map.keys()))

observation_df = observation_df.select('ssm_id',struct(*normal_genotype_map.keys()).alias('normal_genotype'),
                                       struct(*tumor_genotype_map.keys()).alias('tumor_genotype'),
                                       struct(*tumor_validation_map.keys()).alias('validation'),
                                       struct(*read_depth_map.keys()).alias('read_depth'),
                                       struct(*input_bam_map.keys()).alias('input_bam_file'),
                                       struct(*sample_map.keys()).alias('sample'),
                                       *maf_observation_map.keys())\
                                    .drop('gene_id')\
                                .drop_duplicates()

In [15]:
#observation_df.printSchema()

### Get case dataframe from existing graph

In [16]:
#doc = requests.get('http://elasticsearch.service.consul:9200/gdc_from_graph_35/_search').json()['hits']['hits'][0]['_source']
case_fields = ['case_id',
               'submitter_id',
               'state',
               'project.*',
               'program.*',
               'exposures.*',
               'demographic.*',
               '*_ids',
               'diagnoses.*',
               'state',
               'diagnoses.created_datetime']

case_df = sqlContext.read.format("es")\
    .option('es.nodes', 'elasticsearch.service.consul')\
    .option('es.read.field.include', ','.join(case_fields))\
    .option('es.read.field.exclude', 'diagnoses.treatments,summary')\
    .option('es.read.field.as.array.include','*_ids')\
    .option('es.resource.read', 'gdc_from_graph/case')\
    .option('es.nodes.resolve.hostname','false')\
    .load("gdc_from_graph")

In [17]:
#case_df.printSchema()

## Assemble constituent parts

Remember what we're shooting for:
```
gene{}
     |___ case[]
             |___ ssm[]
                   |___ consequence[]
                   |             |_____ transcript{}
                   |                          |_____ annotation{}
                   |___ observation[]
```

In [18]:
#tran_anno_df.limit(2).show()

### Merge annotation with transcript

In [19]:
cons_tran_anno_df = tran_anno_df.select(struct(struct(*tran_anno_df.drop('gene_id').drop('ssm_id').columns).alias('transcript')).alias('consequence'),'ssm_id')\
                                .groupBy('ssm_id')\
                                .agg(collect_list('consequence').alias('consequence'))

In [20]:
#cons_tran_anno_df.printSchema()

### Join observation with consequence

In [21]:
#observation_df.persist().count()
#cons_tran_anno_df.persist().count()

In [22]:
import random
def salt(key, doc_count=0):
    return str(random.randint(0,int(max(0,doc_count-1024)**5)))+key
salt_udf = udf(salt, StringType())

In [23]:
salted_observation_df = observation_df.withColumn('salt_key', salt_udf(col('ssm_id')))\
                                     #.repartition(1024, 'salt_key')
#salted_observation_df.persist().count()

In [24]:
salted_consequence_df = cons_tran_anno_df.withColumn('salt_key', salt_udf(col('ssm_id')))
                                         #.repartition(64, 'salt_key')
#salted_consequence_df.persist().count()

In [25]:
# Partitions before and after salting
#pdf = salted_observation_df.groupBy('gene_symbol').agg(count('gene_symbol')).toPandas()
#pdf.plot()
#pdf = salted_observation_df.groupBy('salt_key').agg(count('gene_symbol')).toPandas()
#pdf.plot()

In [26]:
cons_obs = salted_consequence_df.join(salted_observation_df, salted_consequence_df.ssm_id == salted_observation_df.ssm_id, 'outer')\
                            .drop(salted_consequence_df.ssm_id)\
                            .drop(salted_consequence_df.salt_key)\
                            .select('ssm_id','consequence',struct(*salted_observation_df.drop('ssm_id').drop('salt_key').columns).alias('observation'))

cons_obs = cons_tran_anno_df.join(observation_df, cons_tran_anno_df.gene_id == observation_df.gene_id, 'outer')\
                            .drop(cons_tran_anno_df.gene_id)\
                            .drop(observation_df.gene_id)\
                            .select('gene_id','consequence',struct(*observation_df.drop('gene_id').columns).alias('observation'))

In [27]:
cons_obs.select()

DataFrame[]

In [28]:
#pdf = cons_obs.groupBy('gene_symbol').agg(count('gene_symbol')).toPandas()

### Join consequence into ssm

In [29]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [30]:
def view_salt(pdf1, pdf2):
    f, axes = plt.subplots(1,2, figsize=(10,4), sharex=False)
    pdf1.columns = ['salt_key', 'count']
    pdf2.columns = ['salt_key', 'count']
    axes[0].plot(pdf1['count'], 'b.')
    axes[0].set_yscale('log')
    axes[0].set_ylim(10)
    axes[0].set_xlim(0,len(pdf1))
    axes[0].set_title('unsalted')
    axes[1].plot(pdf2['count'], 'g.')
    axes[1].set_xlim(0,len(pdf2))
    axes[1].set_title('salted')
    plt.show()

In [31]:
#pdf = cons_obs.select('gene_symbol', size('consequence')).toPandas()

In [32]:
#pdf.max()

In [33]:
# Salt consequence-observation
salted_cons_obs = cons_obs.withColumn('doc_count', size(col('consequence')))\
                            .withColumn('salt_key', salt_udf(col('ssm_id'), col('doc_count')))
#salted_cons_obs.repartition(1024, 'salt_key').count()

In [34]:
#pdf = salted_cons_obs.groupBy('salt_key').agg(sum('doc_count').alias('sum')).toPandas()
#pdf.sort_values('sum', ascending=False).reset_index(drop=True).plot()
#plt.gca().set_yscale('log')

In [35]:
#pdf = salted_cons_obs.groupBy('salt_key').agg(sum('doc_count').alias('sum')).toPandas()
#pdf.sort_values('sum', ascending=False).reset_index(drop=True).plot()
#plt.gca().set_yscale('log')

In [36]:
#pdf1 = cons_obs.groupBy('gene_symbol').agg(count('gene_symbol')).persist().toPandas()
#pdf2 = salted_cons_obs.groupBy('salt_key').agg(count('gene_symbol')).persist().toPandas()
#view_salt(pdf1,pdf2)

In [37]:
salted_ssm = ssm_df.withColumn('salt_key', salt_udf(col('gene_id')))
#salted_ssm.repartition('salt_key').persist().count()

In [38]:
#pdf1 = ssm_df.groupBy('gene_symbol').agg(count('gene_symbol')).persist().toPandas()
#pdf2 = salted_ssm.groupBy('salt_key').agg(count('gene_symbol')).persist().toPandas()
#view_salt(pdf1,pdf2)

In [39]:
sqlContext.sql("set spark.sql.shuffle.partitions=2048")

DataFrame[key: string, value: string]

In [40]:
#pd_df = ssm_cons.select(size('ssm')).toPandas()

In [41]:
#pd_df.head()

In [42]:
ssm_cons = salted_cons_obs.join(salted_ssm, salted_ssm.ssm_id == salted_cons_obs.ssm_id, 'left')\
                        .drop(salted_cons_obs.ssm_id)\
                        .drop(salted_cons_obs.salt_key)\
                        .select('_case_submitter_id', struct('consequence','observation',*ssm_df.drop('_case_submitter_id').drop('gene_id').columns).alias('ssm'))\
                        .groupBy('_case_submitter_id')\
                        .agg(collect_list('ssm').alias('ssm'))

ssm_cons = cons_obs.join(ssm_df, ssm_df.gene_id == cons_obs.gene_id, 'left')\
                        .drop(cons_obs.gene_id)\
                        .select('_case_submitter_id', struct('consequence','observation',*ssm_df.columns).alias('ssm'))\
                        .groupBy('_case_submitter_id')\
                        .agg(collect_list('ssm').alias('ssm'))

In [43]:
#ssm_cons.select(size('ssm')).describe().show()

### Join ssm with case

In [44]:
case_ssm = case_df.join(ssm_cons, case_df.submitter_id == ssm_cons._case_submitter_id, 'left')\
                    .drop(ssm_cons._case_submitter_id)\
                    .select('submitter_id', struct('ssm', *case_df.columns).alias('case'))

In [45]:
#case_ssm.persist().count()

In [46]:
#case_ssm.columns

In [47]:
#gene_df.drop_duplicates().count()

### Join case with gene

gene_centric = gene_df.join(case_ssm, gene_df._case_submitter_id == case_ssm.submitter_id, 'left')\
                    .groupBy('_case_submitter_id')\
                    .agg(collect_list('case').alias('case'))\
                    .join(gene_df, 'gene_id' == gene_df.gene_id, 'left')\
                    .drop(gene_df.gene_id)\
                    .drop('_case_submitter_id')

In [48]:
#gene_df = gene_df.where(gene_df.gene_id == 'ENSG00000005471')
gene_centric = gene_df.join(case_ssm, gene_df._case_submitter_id == case_ssm.submitter_id, 'inner')\
                    .groupBy(*gene_df.columns).agg(collect_list('case').alias('case'))\
                    .drop('_case_submitter_id')

In [49]:
#gene_centric.printSchema()

In [50]:
#gene_centric.persist().count()

In [51]:
#gene_centric.count()
#460578

In [52]:
#gene_centric.select(max(size('case'))).show()

## Export df to es

In [53]:
index = 'dan-r2-gene'

#### New vis index

In [54]:
%autoreload
from exports.mappings import GeneMapper
m = GeneMapper()
m.mapping['_size'] =  {"enabled": 'true'}
#m.mapping['properties']['case'].keys()
#m.mapping['dynamic'] = 'true'
#m.mapping['properties']['case']['dynamic'] = 'true'
#m.mapping['properties']['case']['properties']['files']['dynamic'] = 'true'
#m.mapping['properties']['case']['properties']['files']['properties']['cases']['dynamic'] = 'true'

In [56]:
import json

print requests.delete('http://elasticsearchvis.service.consul:9200/{}'.format(index)).json()

data = json.dumps({"settings":{"index":{
                "refresh_interval":"10m",
                "number_of_shards":20,
                "number_of_replicas":1,
                "mapper.dynamic":False,
                "mapping.nested_fields.limit":100,
                "mapping.total_fields.limit":2000,
                    "translog": {
                    "durability": "async",
                    "sync_interval": "15s"
                }
            }},"mappings":{
                "gene":m.mapping
            }})
#print requests.put('http://localhost:9200/test/', data=data).json()
print requests.put('http://elasticsearchvis.service.consul:9200/{}'.format(index), data=data).json()

{u'acknowledged': True}
{u'acknowledged': True, u'shards_acknowledged': True}


In [ ]:
#%%time
# Stop index refreshing while we bulk load
gene_centric.limit(1).coalesce(2048).write.format('org.elasticsearch.spark.sql')\
                    .option('es.nodes', 'elasticsearchvis.service.consul')\
                    .option('es.nodes.resolve.hostname','false')\
                    .option('es.resource.write', '{}/gene'.format(index))\
                    .option('es.http.timeout', '10m')\
                    .option('es.batch.write.retry.count','-1')\
                    .option('es.batch.write.retry.wait', '30s')\
                    .option('es.batch.size.bytes','100mb')\
                    .option('es.batch.size.entries', '1000')\
                    .option('es.mapping.id','gene_id')\
                    .save('{}/gene'.format(index))
            
#requests.post('http://localhost:9200/gene/_refresh')
# 6min 30s to write with defaults
# 5min 37s to write with batchsize = 1mb
# 5min 13s to write with batchsize = 512kb

In [ ]:
data = {
    "actions" : [
        { "add" : { "index" : index, "alias" : "gene-centric" } }
    ]
}

print requests.post('http://elasticsearchvis.service.consul:9200/_aliases', data=json.dumps(data)).json()

In [ ]:
test_query = {
  "query": {
    "nested": {
      "path": "case",
      "score_mode": "sum",
      "query": {
        "function_score": {
          "query": {
            "bool": {
              "must": [
                {
                "terms": {
                  "case.project.project_id": [
                    "TCGA-ACC"
                  ]
                }
                }
              ]
            }
          }
        }
      }
    }
  }
}
      
len(requests.post('http://localhost:9200/test/_search', data=json.dumps(test_query)).json()['hits']['hits'])